##### 01 - Data Understanding


In [1]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')
TXN_FILE = RAW / 'card_transaction.v1.csv'
USERS_FILE = RAW / 'sd254_users.csv'
CARDS_FILE = RAW / 'sd254_cards.csv'

In [2]:
users = pd.read_csv(USERS_FILE)
cards = pd.read_csv(CARDS_FILE)

print('Users: ', users.shape)
print('Cards: ', cards.shape)
print(users.dtypes)
print(cards.dtypes)

Users:  (2000, 18)
Cards:  (6146, 13)
Person                             str
Current Age                      int64
Retirement Age                   int64
Birth Year                       int64
Birth Month                      int64
Gender                             str
Address                            str
Apartment                      float64
City                               str
State                              str
Zipcode                          int64
Latitude                       float64
Longitude                      float64
Per Capita Income - Zipcode        str
Yearly Income - Person             str
Total Debt                         str
FICO Score                       int64
Num Credit Cards                 int64
dtype: object
User                     int64
CARD INDEX               int64
Card Brand                 str
Card Type                  str
Card Number              int64
Expires                    str
CVV                      int64
Has Chip                   st

In [3]:
columns = pd.read_csv(TXN_FILE, nrows=0).columns
total_rows = 0
blank_counts = pd.Series(0, index=columns)

for chunk in pd.read_csv(TXN_FILE, dtype=str, keep_default_na=False, 
                         chunksize=200_000):
    total_rows += len(chunk)
    blank_counts += (chunk == '').sum()

print(f'Total rows: {total_rows:,}')
print(blank_counts)

Total rows: 24,386,900
User                     0
Card                     0
Year                     0
Month                    0
Day                      0
Time                     0
Amount                   0
Use Chip                 0
Merchant Name            0
Merchant City            0
Merchant State     2720821
Zip                2878135
MCC                      0
Errors?           23998469
Is Fraud?                0
dtype: int64


In [4]:
from collections import Counter

use_chip = Counter()
is_fraud = Counter()
errors = Counter()
years = Counter()
state_lengths = Counter()
online_rows = 0
negative_amounts = 0
mcc_codes = set()

for chunk in pd.read_csv(TXN_FILE, keep_default_na=False, chunksize=200_000):
    use_chip.update(chunk['Use Chip'])
    is_fraud.update(chunk['Is Fraud?'])
    errors.update(chunk['Errors?'])
    years.update(chunk['Year'])
    state_lengths.update(chunk['Merchant State'].str.len())
    online_rows += (chunk['Merchant City'].str.strip() == 'ONLINE').sum()
    negative_amounts += (chunk['Amount'].str.contains('-').sum())
    mcc_codes.update(chunk['MCC'].unique())

print("Use Chip:", dict(use_chip))
print("Is Fraud?:", dict(is_fraud))
print("Years:", min(years), "to", max(years), "| distinct:", len(years))
print("Online rows (city text):", online_rows)
print("Negative amounts:", negative_amounts)
print("Distinct MCC:", len(mcc_codes))
print("Merchant State length counts:", dict(state_lengths))
print("Distinct Errors? values:", len(errors))
for value, count in errors.most_common(15):
    print(repr(value), count)

Use Chip: {'Swipe Transaction': 15386082, 'Online Transaction': 2713220, 'Chip Transaction': 6287598}
Is Fraud?: {'No': 24357143, 'Yes': 29757}
Years: 1991 to 2020 | distinct: 30
Online rows (city text): 2720821
Negative amounts: 1244689
Distinct MCC: 109
Merchant State length counts: {2: 21508765, 0: 2720821, 11: 10053, 7: 14527, 5: 26681, 6: 81051, 18: 2754, 8: 5060, 12: 1041, 9: 3691, 14: 8796, 4: 1009, 10: 1782, 16: 42, 20: 552, 13: 87, 15: 31, 21: 16, 32: 12, 22: 60, 17: 3, 19: 29, 24: 37}
Distinct Errors? values: 24
'' 23998469
'Insufficient Balance,' 242783
'Bad PIN,' 58918
'Technical Glitch,' 48157
'Bad Card Number,' 13321
'Bad CVV,' 10740
'Bad Expiration,' 10716
'Bad Zipcode,' 2079
'Bad PIN,Insufficient Balance,' 581
'Insufficient Balance,Technical Glitch,' 457
'Bad PIN,Technical Glitch,' 128
'Bad Card Number,Insufficient Balance,' 122
'Bad CVV,Insufficient Balance,' 89
'Bad Expiration,Insufficient Balance,' 78
'Bad Card Number,Bad CVV,' 60


In [5]:
state_other = Counter()
chip_fraud = Counter()
negative_fraud = Counter()
error_fraud = Counter()
online_chip = Counter()
error_types = set()

for chunk in pd.read_csv(TXN_FILE, dtype=str, keep_default_na=False, chunksize=200_000):
    long_states = chunk.loc[chunk["Merchant State"].str.len() > 2, "Merchant State"]
    state_other.update(long_states)

    chip_fraud.update(zip(chunk["Use Chip"], chunk["Is Fraud?"]))

    is_negative = chunk["Amount"].str.contains("-")
    negative_fraud.update(chunk.loc[is_negative, "Is Fraud?"])

    has_error = chunk["Errors?"] != ""
    error_fraud.update(chunk.loc[has_error, "Is Fraud?"])

    is_online_city = chunk["Merchant City"].str.strip() == "ONLINE"
    online_chip.update(chunk.loc[is_online_city, "Use Chip"])

    for text in chunk["Errors?"].unique():
        for part in text.split(","):
            if part.strip() != "":
                error_types.add(part.strip())

print("Longer state values (top 15):", state_other.most_common(15))
print("Distinct longer state values:", len(state_other))
print("Use Chip x Is Fraud?:", dict(chip_fraud))
print("Fraud flag among negative amounts:", dict(negative_fraud))
print("Fraud flag among rows with an error:", dict(error_fraud))
print("Use Chip among rows with city ONLINE:", dict(online_chip))
print("Base error types:", sorted(error_types))

Longer state values (top 15): [('Mexico', 47152), ('Canada', 20148), ('Italy', 8730), ('United Kingdom', 8055), ('France', 5311), ('Germany', 5263), ('China', 4996), ('Japan', 3955), ('India', 3482), ('Spain', 3203), ('Dominican Republic', 2754), ('Netherlands', 2616), ('Jamaica', 2184), ('South Korea', 2178), ('Switzerland', 1947)]
Distinct longer state values: 171
Use Chip x Is Fraud?: {('Swipe Transaction', 'No'): 15379510, ('Online Transaction', 'No'): 2694871, ('Chip Transaction', 'No'): 6282762, ('Online Transaction', 'Yes'): 18349, ('Chip Transaction', 'Yes'): 4836, ('Swipe Transaction', 'Yes'): 6572}
Fraud flag among negative amounts: {'No': 1243564, 'Yes': 1125}
Fraud flag among rows with an error: {'No': 387145, 'Yes': 1286}
Use Chip among rows with city ONLINE: {'Online Transaction': 2713220, 'Chip Transaction': 7601}
Base error types: ['Bad CVV', 'Bad Card Number', 'Bad Expiration', 'Bad PIN', 'Bad Zipcode', 'Insufficient Balance', 'Technical Glitch']


#### Keys and formats

**Questions:**
1. Is the user ID in transactions and cards simply the row position in the users file?
2. Does every (User, Card) pair in the transactions exist in the cards table?
3. Are Time, Amount, dates and Zip consistently formatted?

In [6]:
users = pd.read_csv(USERS_FILE)
cards = pd.read_csv(CARDS_FILE)

print("Duplicate Person values in users:", users["Person"].duplicated().sum())
print("Duplicate (User, CARD INDEX) in cards:", cards.duplicated(["User", "CARD INDEX"]).sum())
print("User ids in cards:", cards["User"].min(), "to", cards["User"].max(), "| distinct:", cards["User"].nunique())


cards_per_user = cards.groupby("User").size()
expected = cards_per_user.reindex(range(len(users)))
matches = (users["Num Credit Cards"].values == expected.values).sum()
print("Users whose Num Credit Cards equals the cards-table count:", matches, "of", len(users))

Duplicate Person values in users: 7
Duplicate (User, CARD INDEX) in cards: 0
User ids in cards: 0 to 1999 | distinct: 2000
Users whose Num Credit Cards equals the cards-table count: 2000 of 2000


In [11]:
pairs = set()
bad_time = 0
bad_amount = 0
bad_date = 0
bad_zip = 0
short_zip = 0

for chunk in pd.read_csv(TXN_FILE, chunksize=200_000, keep_default_na=False, dtype=str):
    pairs.update(zip(chunk['User'], chunk['Card']))

    bad_time += (~chunk["Time"].str.match(r"^\d{2}:\d{2}$")).sum()
    bad_amount += (~chunk["Amount"].str.match(r"^-?\$\d+\.\d{2}$")).sum()

    dates = pd.to_datetime(chunk[["Year", "Month", "Day"]].astype(int), errors="coerce")
    bad_date += dates.isna().sum()

    zips = chunk.loc[chunk["Zip"] != "", "Zip"]
    bad_zip += (~zips.str.match(r"^\d+\.0$")).sum()
    short_zip += (zips.str.len() < 7).sum()

card_keys = set(zip(cards["User"].astype(str), cards["CARD INDEX"].astype(str)))

print("Distinct (User, Card) pairs in transactions:", len(pairs))
print("Pairs missing from the cards table:", len(pairs - card_keys))
print("Cards with no transactions:", len(card_keys - pairs))
print("Bad Time:", bad_time, "| bad Amount:", bad_amount, "| invalid dates:", bad_date)
print("Zip not like 12345.0:", bad_zip, "| zips shorter than 5 digits:", short_zip)

Distinct (User, Card) pairs in transactions: 6139
Pairs missing from the cards table: 0
Cards with no transactions: 7
Bad Time: 0 | bad Amount: 1244689 | invalid dates: 0
Zip not like 12345.0: 0 | zips shorter than 5 digits: 1555116


In [12]:
samples = []
max_amount = 0.0
min_amount = 0.0
hours = Counter()
per_year = Counter()

for chunk in pd.read_csv(TXN_FILE, dtype=str, keep_default_na=False, chunksize=200_000):
    if len(samples) < 5:
        negatives = chunk.loc[chunk["Amount"].str.contains("-"), "Amount"]
        samples.extend(negatives.head(5).tolist())

    amounts = chunk["Amount"].str.replace("$", "", regex=False).astype(float)
    max_amount = max(max_amount, amounts.max())
    min_amount = min(min_amount, amounts.min())

    hours.update(chunk["Time"].str[:2])
    per_year.update(chunk["Year"])

print("Sample negative amounts:", samples[:5])
print("Amount range:", min_amount, "to", max_amount)
print("Hours seen:", sorted(hours))
print("Transactions per year:", sorted(per_year.items()))

Sample negative amounts: ['$-99.00', '$-100.00', '$-99.00', '$-88.00', '$-207.00']
Amount range: -500.0 to 12390.5
Hours seen: ['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23']
Transactions per year: [('1991', 1585), ('1992', 5134), ('1993', 8378), ('1994', 14316), ('1995', 20928), ('1996', 29945), ('1997', 49753), ('1998', 78345), ('1999', 118250), ('2000', 177729), ('2001', 257998), ('2002', 350732), ('2003', 466408), ('2004', 597003), ('2005', 746653), ('2006', 908793), ('2007', 1064483), ('2008', 1223460), ('2009', 1355434), ('2010', 1491225), ('2011', 1570551), ('2012', 1610829), ('2013', 1650917), ('2014', 1672343), ('2015', 1701371), ('2016', 1708924), ('2017', 1723360), ('2018', 1721615), ('2019', 1723938), ('2020', 336500)]
